# Probe a text model

A test split tells you the average. A probe tells you the *shape* — negation,
length, spelling, sarcasm — by sending inputs you wrote on purpose and reading
what the model does with them. It is the cheapest way to find a failure mode
that no metric names.

In [ ]:
import orinth

model = next(
    entry
    for entry in orinth.models.list()
    if entry.available and entry.task_type == "text_classification"
)
print(model)
predictor = orinth.models.predictor(model.id)
params = orinth.models.parameters()

## A probe set, written by hand

Grouped by what each line is testing. The grouping is the analysis — a flat list
of twelve sentences tells you twelve facts, and this tells you four.

In [ ]:
PROBES = {
    "plain": [
        "the battery lasts all day and charging is quick",
        "the screen cracked after a week",
    ],
    "negation": [
        "this is not a bad product at all",
        "I would not say I am unhappy with it",
    ],
    "length": [
        "fine",
        "it works exactly as described and I have had no issues whatsoever in the three "
        "months since it arrived, which is more than I can say for the two I owned before it",
    ],
    "noise": [
        "the batttery draaains realy fast!!!",
        "THE BATTERY DRAINS REALLY FAST",
    ],
}

In [ ]:
import polars as pl

rows = []
for group, samples in PROBES.items():
    for text in samples:
        answer = predictor.predict_text(text, params)
        scores = answer.get("scores") or {}
        top = max(scores, key=scores.get) if scores else answer.get("label", "")
        rows.append(
            {
                "probe": group,
                "text": text[:52] + ("…" if len(text) > 52 else ""),
                "label": answer.get("label", top),
                "confidence": round(float(scores.get(top, 0.0)), 3),
            }
        )

probe = pl.DataFrame(rows)
probe

## Where confidence collapses

A model that answers `negation` at chance is doing keyword matching. One whose
confidence drops on `noise` is brittle to exactly the input real users type.

In [ ]:
probe.group_by("probe").agg(
    pl.col("confidence").mean().round(3).alias("mean_confidence"),
    pl.col("label").n_unique().alias("distinct_answers"),
).sort("mean_confidence")

## The same inputs, recorded

`orinth.inference.predict(text=…)` is the recorded version: each answer gets an
id and a row on `/inference`, which is what you want when the probe is evidence
rather than a look.

In [ ]:
recorded = orinth.inference.predict(model.id, text=PROBES["negation"][0])
print(recorded.id, "|", recorded.label, "|", recorded.text)

## Keep it

A probe set is worth more the second time you run it. Logged as a run, "did the
retrain fix negation" becomes a question with an answer.

In [ ]:
with orinth.runs.start(f"probe {model.id}", params={"probes": len(rows)}) as run:
    for group, frame in probe.group_by("probe"):
        run.log(**{f"confidence_{group[0]}": float(frame["confidence"].mean())})
    run.log_artifact(probe.write_csv().encode("utf-8"), name="probe.csv")